# Building a Chat Model Chain with LangChain

This notebook explores how LangChain connects a chat model to a reusable prompt template.

The progression is:

1. Load a chat model from configuration.
2. Invoke the model directly with a plain-text question.
3. Create a `ChatPromptTemplate` with named variables.
4. Compose the prompt and model into a chain using the `|` operator.
5. Invoke the chain by providing values for the template variables.
6. Extend the prompt to use multiple variables.

The key idea is that a chain separates **prompt construction** from **model execution**, making the same model interaction easier to reuse with different inputs.

> The code expects the required environment variables and dependencies to be configured before the notebook is run.

In [4]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model, BaseChatModel

load_dotenv()

model_name ="google_genai:gemini-3.5-flash-lite"

In [5]:
model = init_chat_model(model_name)

In [18]:
from langchain.messages import  HumanMessage
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages(
    
    [
        ("human","What is the capital of {country}?")
    ]
)

print(prompt)

input_variables=['country'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='What is the capital of {country}?'), additional_kwargs={})]


## Create a reusable prompt template

Instead of hard-coding a complete question every time, the prompt uses the `{country}` placeholder. The template can later be formatted with different country names and passed to the model.

In [15]:
model.invoke("What is the capital of france?")

AIMessage(content=[{'type': 'text', 'text': 'The capital of France is Paris.', 'extras': {'signature': 'El4KXAFpFH0Tc/vVPhyvNdOMX1m0V4m8JeKfYubUEcLJdurIdR3LvQmiL/T+3eCPg3bWmFb9g61ik+LCEkTORDApBlEmWa7FPPcs0Ji9v+Zbzycd7IOK1K/EVK775ye0'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0bf2b-9323-78c1-9cc2-d19470caeb38-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})

In [12]:
chain = prompt | model

## Compose the chain

The `|` operator creates a `RunnableSequence`: the prompt receives the input first, and its formatted messages are then passed to the chat model.

This lets us call the complete workflow with one `.invoke()` call.

In [13]:
type(chain  )

langchain_core.runnables.base.RunnableSequence

In [19]:
chain.invoke({'country': 'France'  })

AIMessage(content=[{'type': 'text', 'text': 'The capital of France is Paris.', 'extras': {'signature': 'El4KXAFpFH0T8gUXSppxt70mN6OVwzsHd9EPOajfAOaCx2JlEaKFCZEfGDWMybeb1QGWsnmfN1yCKNPcZINDcMSgEc9okyrrK2LoUcp7kY3tGpdlMxKmUB/LQiV5lUHe'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0bf2e-01c1-7a51-9545-0322b73ee52b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})

In [22]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human","What is the capital of {country}?"),
        ("human","Explain its signifcance in {topic}")
    ]
)
chain = prompt | model
chain.invoke({'country': 'India', 'topic': 'Ancient history'  })

AIMessage(content=[{'type': 'text', 'text': 'The capital of India is **New Delhi**. \n\nWhile New Delhi is the modern seat of government (constructed as the imperial capital by the British in the early 20th century), it is part of the broader National Capital Territory of Delhi, a region that has been the heart of Indian history for millennia. \n\nHere is an explanation of the significance of Delhi (and its ancient precursors) in **ancient Indian history**:\n\n### 1. The Strategic Gateway (Geography and Geopolitics)\nDelhi’s significance in ancient history is primarily due to its geography. Located in north-central India, Delhi sits in the narrow corridor between the Himalayas and the Thar Desert. \n* **The Yamuna River:** It is situated on the banks of the Yamuna River, providing water and fertile soil.\n* **The Crossroads of Empire:** It served as the gateway between the fertile Gangetic plain to the east and the Indus river basin and mountain passes of the northwest (through which i

## Use multiple prompt variables

A prompt can contain more than one placeholder. This example asks for a country's capital and then asks the model to explain that capital's significance within a supplied topic.

The dictionary passed to `.invoke()` must provide a value for every placeholder: `country` and `topic`.